In [7]:
import pandas as pd
import numpy as np
from xgboost import XGBClassifier
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV
from sklearn.model_selection import StratifiedKFold
from sklearn.base import BaseEstimator
from data_preprocessing import create_train_test_val_sets, read_processed_data


In [16]:
#Create test train splits
x_mendeley, y_mendeley = read_processed_data(r"..\data\processed\mendeley_processed.csv")
x_phiusiil, y_phiusiil= read_processed_data(r"..\data\processed\phiusiil_processed.csv")

mendeley_sets = create_train_test_val_sets(x_mendeley,y_mendeley, label_col="Label", test_size=0.2, n_splits=5)
phiusiil_sets = create_train_test_val_sets(x_phiusiil,y_phiusiil, label_col="Label", test_size=0.2, n_splits=5)

Train/validation/test split prepared: 198360 instances for training and validation, 49590 instances for testing
Stratified 5-fold CV splits created.
Train/validation/test split prepared: 188636 instances for training and validation, 47159 instances for testing
Stratified 5-fold CV splits created.


### Tuning Classifiers

In [28]:
#XGBoost
def optimize_xgboost(X: pd.DataFrame, y: pd.Series, dataset: str, splits) -> RandomizedSearchCV:
    scale_weights = [1.0]
    counts = y.value_counts(normalize=True)
    if dataset == 'mendeley':
        scale_weights.append(counts[1]/counts[0])
    elif dataset == 'phiusiil':
        scale_weights.append(counts[0]/counts[1])
    
    params = {
        'max_depth': [4, 5, 6, 8, 10],
        'min_child_weight': [1, 3, 5, 7],
        'gamma': [0, 0.1, 0.2, 0.4],
        'subsample': [0.6, 0.7, 0.8, 0.9, 1.0],
        'colsample_bytree': [0.6, 0.7, 0.8, 0.9, 1.0],
        'learning_rate': [0.01, 0.05, 0.1, 0.2, 0.4],
        'n_estimators': [100, 300, 500, 750, 1000],
        'scale_pos_weight': scale_weights
    }

    xgb = XGBClassifier(random_state=42)
    random_search = RandomizedSearchCV(xgb, param_distributions=params, random_state=42, cv=splits)
    random_search.fit(X, y)

    print('\n Best hyperparameters:')
    print(random_search.best_params_)

    return random_search

print("Running hyperparameter tuning using Mendeley Dataset:")
xgboost_mendeley = optimize_xgboost(mendeley_sets["x_train_val"], mendeley_sets["y_train_val"], 'mendeley', mendeley_sets["cv_splits"])
# xgboost_mendeley = XGBClassifier(subsample=0.7, scale_pos_weight=0.93, n_estimators=750, min_child_weight=3, max_depth=10, learning_rate=0.4, gamma=0.2, colsample_bytree=0.8)
print("Running hyperparameter tuning using Phiusiil Dataset:")
xgboost_phiusiil = optimize_xgboost(phiusiil_sets["x_train_val"], phiusiil_sets["y_train_val"], 'phiusiil', phiusiil_sets["cv_splits"])
# xgboost_phiusiil = XGBClassifier(subsample=0.9, n_estimators=300, min_child_weight=3, max_depth=5, learning_rate=0.01, gamma=0, colsample_bytree=0.9)


Running hyperparameter tuning using Mendeley Dataset:

 Best hyperparameters:
{'subsample': 0.7, 'scale_pos_weight': np.float64(0.9289527680802856), 'n_estimators': 750, 'min_child_weight': 3, 'max_depth': 10, 'learning_rate': 0.4, 'gamma': 0.2, 'colsample_bytree': 0.8}
Running hyperparameter tuning using Phiusiil Dataset:

 Best hyperparameters:
{'subsample': 0.9, 'scale_pos_weight': 1.0, 'n_estimators': 300, 'min_child_weight': 3, 'max_depth': 5, 'learning_rate': 0.01, 'gamma': 0, 'colsample_bytree': 0.9}


In [ ]:
from sklearn.metrics import classification_report
def train_no_feature_selection(dataset, model):
    y_pred = model.predict(dataset["x_test"])
    print('Confusion Matrix:')
    print(classification_report(dataset["y_test"], y_pred))
    

print('Mendeley Results:')
train_no_feature_selection(mendeley_sets, xgboost_mendeley.best_estimator_)

print('Phiusiil Results:')
train_no_feature_selection(phiusiil_sets, xgboost_phiusiil.best_estimator_)



Mendeley Results:
Confusion Matrix:
              precision    recall  f1-score   support

           0       0.95      0.96      0.96     25708
           1       0.96      0.95      0.95     23882

    accuracy                           0.95     49590
   macro avg       0.95      0.95      0.95     49590
weighted avg       0.95      0.95      0.95     49590

Phiusiil Results:
Confusion Matrix:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00     20189
           1       1.00      1.00      1.00     26970

    accuracy                           1.00     47159
   macro avg       1.00      1.00      1.00     47159
weighted avg       1.00      1.00      1.00     47159

